# 07 · Live HTTP server (`.tp.serve`)

`da.tp.serve(port=...)` starts a tiny FastAPI/uvicorn server exposing the field for a browser to `fetch()`:

- `GET /field` → the JSON payload
- `GET /health` → `{"status": "ok"}`
- `WS  /ws` → pushes the payload on connect

`serve()` **blocks**, so in a notebook we run it in a daemon thread (it dies with the kernel).

> Requires `pip install 'pyterraplot[serve]'`.

In [ ]:
import numpy as np
import xarray as xr
import pyterraplot  # registers the .tp accessor on DataArray and Dataset

def make_field(nlat=73, nlon=144, phase=0.0, name="t2m",
               long_name="2m temperature anomaly", units="K", holes=True):
    """A smooth, globe-shaped synthetic field on a regular lat/lon grid."""
    lats = np.linspace(90, -90, nlat)
    lons = np.linspace(-180, 180, nlon)
    LON, LAT = np.meshgrid(lons, lats)
    data = (
        8 * np.cos(np.radians(LAT)) * np.sin(np.radians(2 * LON) + phase)
        + 5 * np.sin(np.radians(3 * LON)) * np.cos(np.radians(2 * LAT))
        + 3 * np.cos(np.radians(5 * LON)) * np.sin(np.radians(LAT))
    ).astype(np.float32)
    if holes:
        rng = np.random.default_rng(0)
        data[rng.random((nlat, nlon)) < 0.02] = np.nan  # NaN "missing" cells
    return xr.DataArray(
        data, dims=["lat", "lon"], coords={"lat": lats, "lon": lons},
        name=name, attrs={"units": units, "long_name": long_name},
    )

da = make_field()
da

## Start the server in a background thread

In [ ]:
import threading, time
PORT = 8799

try:
    import uvicorn, fastapi  # noqa: F401
    have_serve = True
except ImportError:
    have_serve = False
    print("pyterraplot[serve] not installed — `pip install 'pyterraplot[serve]'` to run this notebook")

if have_serve:
    t = threading.Thread(target=da.tp.serve, kwargs={"port": PORT}, daemon=True)
    t.start()
    time.sleep(2.0)  # give uvicorn a moment to bind
    print("server thread alive:", t.is_alive())

## Hit the endpoints

Using stdlib `urllib` so there are no extra deps.

In [ ]:
import json, urllib.request

def get(path):
    with urllib.request.urlopen(f"http://127.0.0.1:{PORT}{path}", timeout=5) as r:
        return json.loads(r.read())

if have_serve:
    print("/health :", get("/health"))
    field = get("/field")
    print("/field keys :", list(field))
    print("/field grid :", len(field["field"]), "×", len(field["field"][0]))

## Browser side

In your terraplot app, point `fetch` at the running server:

```javascript
import { GeoSphere } from 'terraplot';
const p = await fetch('http://127.0.0.1:8799/field').then(r => r.json());
new GeoSphere('#map').pcolormesh(p.lons, p.lats, p.field, { cmap: 'RdYlBu_r' });
```

The daemon thread shuts down automatically when the kernel stops. To free the port sooner, restart the kernel.